# Player Retention Analytics

This notebook demonstrates a lightweight game analytics workflow using a mock player event dataset.

It covers:
- loading player event data
- calculating Daily Active Users (DAU)
- calculating D1 retention
- visualizing player activity trends


In [ ]:
import os
import pandas as pd
import duckdb
import matplotlib.pyplot as plt


In [ ]:
# Resolve project paths
BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'data', 'player_events_mock_v2.csv')

df = pd.read_csv(DATA_PATH)
df.head()


## Daily Active Users (DAU)

DAU is defined as the number of distinct active players per calendar day.


In [ ]:
con = duckdb.connect()
con.register('player_events', df)

dau = con.execute("""
SELECT
    DATE(event_time) AS date,
    COUNT(DISTINCT player_id) AS dau
FROM player_events
GROUP BY date
ORDER BY date
""").df()

dau.head()


## D1 Retention

D1 retention measures the percentage of players who return and start a session one day after installation.


In [ ]:
d1_retention = con.execute("""
WITH installs AS (
    SELECT DISTINCT player_id, DATE(install_date) AS install_date
    FROM player_events
),
returns_d1 AS (
    SELECT DISTINCT i.player_id, i.install_date
    FROM installs i
    JOIN player_events e
      ON i.player_id = e.player_id
     AND DATE(e.event_time) = i.install_date + INTERVAL '1 day'
     AND e.event_name = 'session_start'
)
SELECT
    i.install_date,
    COUNT(DISTINCT i.player_id) AS installers,
    COUNT(DISTINCT r.player_id) AS retained,
    ROUND(COUNT(DISTINCT r.player_id) * 1.0 / COUNT(DISTINCT i.player_id), 3) AS d1_retention
FROM installs i
LEFT JOIN returns_d1 r
  ON i.player_id = r.player_id
 AND i.install_date = r.install_date
GROUP BY i.install_date
ORDER BY i.install_date
""").df()

d1_retention.head()


## Visualization

A simple DAU trend chart helps illustrate how player activity changes over time.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(dau['date'], dau['dau'])
plt.xticks(rotation=45)
plt.title('DAU Trend')
plt.xlabel('Date')
plt.ylabel('DAU')
plt.tight_layout()
plt.show()


## Key Takeaways

- DAU provides a high-level view of player activity trends.
- D1 retention helps evaluate early engagement and onboarding effectiveness.
- This workflow mirrors a common game analytics process using SQL + Python.
